In [2]:
#скачивает из свс и загружаем базу

In [3]:
import pandas as pd

df = pd.read_csv('Данные для тестового задания - Данные об аудитории.csv')
df.head()

,date,user_id,view_adverts
0,2023-11-11,8c020470-8461-11ed-83d0-552e8cc749d6,13
1,2023-11-18,5875f070-7b92-11ee-a6fb-8b298e83f4f7,14
2,2023-11-29,3c2d27c0-4fd6-11eb-b89f-2ffb31b67dd6,21
3,2023-11-29,234a96d0-ad16-11ed-a2e6-793ddfeeba1f,23
4,2023-11-29,4d07c180-644f-11eb-879c-b7c02edf4f37,12


In [4]:
df.info()
df.head()
df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16814 entries, 0 to 16813
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   date          16814 non-null  object
 1   user_id       16814 non-null  object
 2   view_adverts  16814 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 394.2+ KB


Index(['date', 'user_id', 'view_adverts'], dtype='object')

In [5]:
#MAU

In [12]:
df['user_id'].nunique()

7639

In [7]:
#DAU

In [8]:
df.groupby('date')['user_id'].nunique().mean()

np.float64(560.4666666666667)

In [9]:
#reterntion

In [10]:
day1 = df[df['date'] == '2023-11-01']['user_id']
day2 = df[df['date'] == '2023-11-02']['user_id']
retention = len(set(day1) & set(day2)) / len(day1)
retention * 100

26.64526484751204

In [13]:
#conversion

In [15]:
df['date'] = pd.to_datetime(df['date'])
nov = df[df['date'].dt.month == 11]
users_with_views = nov[nov['view_adverts'] > 0]['user_id'].nunique()
all_users = nov['user_id'].nunique()
conversion = users_with_views / all_users * 100
print(f'Конверсия: {conversion:.1f}%')

Конверсия: 46.3%


In [16]:
#average view

In [17]:
avg_views = (
    nov.groupby('user_id')['view_adverts']
    .sum()
    .mean()
)
print(f'Среднее кол просмотров: {avg_views:.1f}')

Среднее кол просмотров: 2.9


In [18]:
#NPS

In [19]:
critics = 500
promoters = 1200
neutrals = 300
total = 2000
nps = ((promoters / total) - (critics / total)) * 100
print(f'NPS = {nps:.0f}%')

NPS = 35%


In [20]:
#ARPU A\B

In [32]:
import pandas as pd
from scipy.stats import ttest_ind
ab = pd.read_csv('Данные для тестового задания - Данные АБ тестов.csv')
print(ab.head())
print(ab.columns)

   experiment_num experiment_group   user_id  revenue
0               1             test     38456      520
1               1          control  13125924      806
2               1          control   9761984        0
3               1             test  11387012      208
4               1             test  18319648      104
Index(['experiment_num', 'experiment_group', 'user_id', 'revenue'], dtype='object')


In [33]:
print(ab['experiment_group'].unique())
print(ab['revenue'].head())
print(ab['revenue'].dtype)

['test' 'control']
0    520
1    806
2      0
3    208
4    104
Name: revenue, dtype: int64
int64


In [35]:
ab.columns = ab.columns.str.strip()
ab['experiment_group'] = ab['experiment_group'].astype(str).str.strip()
ab['revenue'] = pd.to_numeric(ab['revenue'], errors='coerce')
experiments = ab['experiment_num'].unique()
for exp in experiments:
    temp = ab[ab['experiment_num'] == exp]
control = temp[temp['experiment_group'] == 'control']['revenue']
test = temp[temp['experiment_group'] == 'test']['revenue']

In [36]:
stat, pvalue = ttest_ind(control, test, equal_var=False)
print(f'Эксперимент {exp}')
print(f'ARPU control = {control.mean():.2f}')
print(f'ARPU test = {test.mean():.2f}')
print(f'p-value = {pvalue:.5f}')

Эксперимент 3
ARPU control = 663.21
ARPU test = 998.67
p-value = 0.06032


In [40]:
if  pvalue < 0.05:
    print('Различия значимы')

    if test.mean() > control.mean():
        print('Рекомендуется внедрить test')
    else:
        print('Test хуже control')

else:
    print('Статистически различий нет')

Статистически различий нет


In [41]:
#average profit

In [44]:
listers = pd.read_csv('Данные для тестового задания - Листеры.csv')
print(listers.head())
print(listers.columns)
avg_income = listers['revenue'].mean()
print(f'\nСредний доход пользователя: {avg_income:.1f}')

   user_id        date  cnt_adverts  age  cnt_contacts  revenue
0      100  2022-01-01            6   21           119       53
1      100  2022-01-02            2   21           200       18
2      100  2022-01-03            6   21           193       42
3      100  2022-01-04            2   21           143       38
4      100  2022-01-05            2   21           190       40
Index(['user_id', 'date', 'cnt_adverts', 'age', 'cnt_contacts', 'revenue'], dtype='object')

Средний доход пользователя: 30.7


In [45]:
#median

In [46]:
median_age = listers['age'].median()
print(f'Медиана возраста: {median_age}')

Медиана возраста: 28.0


In [49]:
#A|B test

In [50]:
from statsmodels.stats.proportion import proportions_ztest
visitors = [100047501, 100001055]
payments = [1003, 1099]
cr_a = payments[0] / visitors[0]
cr_b = payments[1] / visitors[1]
print('\nCR группы A:', round(cr_a * 100, 6), '%')
print('CR группы B:', round(cr_b * 100, 6), '%')


CR группы A: 0.001003 %
CR группы B: 0.001099 %


In [51]:
stat, pvalue = proportions_ztest(payments, visitors)
print(f'p-value = {pvalue:.10f}')
if pvalue < 0.05:
    print('Различия значимы')

    if cr_b > cr_a:
        print('Рекомендуется внедрить вариант B')
    else:
        print('не рекомендуем внедрять вариант В')

else:
    print('Статистически различий нет')

p-value = 0.0353304454
Различия значимы
Рекомендуется внедрить вариант B
